# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print("\nAvailable metadata fields:")
pprint.pprint(metadata.to_json().keys())

## 2. Data Overview
Review available record sets, fields, and their IDs. We list the `@id` for each record set and fields contained. This will help us explore the dataset structure and reference components precisely.


In [ ]:
# List all record sets by their @id
print("Record Sets:")
for rs in dataset.record_sets:
    print(f"  - {rs['@id']}: {rs['name'] if 'name' in rs else ''}")
    if 'field' in rs:
        fields = rs['field']
        # Ensure fields is a list
        if isinstance(fields, dict):
            fields = [fields]
        print("    Fields:")
        for f in fields:
            if isinstance(f, dict):
                f_id = f.get('@id', f)
            else:
                f_id = f
            print(f"      - {f_id}")
    print()

# For demo purposes, pick the first record set's @id
record_sets_ids = [rs['@id'] for rs in dataset.record_sets]
first_record_set_id = record_sets_ids[0] if record_sets_ids else None
print(f"Selected record set for demonstration: {first_record_set_id}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview, always referencing by their `@id` per Croissant best practice.

In [ ]:
# Extract data from each record set
dataframes = {}
for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    # Build DataFrame only if any records are present
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
    else:
        print(f"No records found for {record_set_id}.")

# Display columns for the first record set with data
for rsid, df in dataframes.items():
    print(f"Columns in record set {rsid}: {df.columns.tolist()}")
    display_df_id = rsid
    break # show only one
if dataframes:
    display(dataframes[display_df_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section shows how to reference fields by `@id` in processing steps.

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Pick a numeric field by exploring columns
used_df = dataframes[display_df_id]
numeric_candidates = used_df.select_dtypes(include=[np.number]).columns.tolist()
if numeric_candidates:
    numeric_field_id = numeric_candidates[0]  # Example: 'gad7_score@field' or similar
    print(f"Numeric field selected: {numeric_field_id}")

    threshold = used_df[numeric_field_id].mean()  # Use mean as threshold for this example
    filtered_df = used_df[used_df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize numeric field
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # Check for a group field (e.g. 'gender@field' or similar string id)
    group_candidates = [col for col in used_df.columns if ('gender' in col.lower() or 'education_level' in col.lower() or 'village' in col.lower())]
    if group_candidates:
        group_field_id = group_candidates[0]
        print(f"\nGrouping by: {group_field_id}")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print("\nGrouped means:")
        print(grouped_df.head())
    else:
        group_field_id = None
else:
    print("No numeric fields found for analysis in this record set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here we plot the distribution of the chosen numeric field and (if available) grouped means by a key attribute.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if used_df is not None and numeric_candidates:
    fig, ax = plt.subplots(figsize=(8,4))
    sns.histplot(used_df[numeric_field_id], bins=20, kde=True, ax=ax)
    ax.set_title(f"Distribution of {numeric_field_id}")
    plt.show()

    # If grouped_df is available
    if 'grouped_df' in locals() and group_field_id:
        fig, ax = plt.subplots(figsize=(10,6))
        sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id, ax=ax)
        ax.set_title(f"Average {numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=30)
        plt.show()

## 6. Conclusion
We have explored the structure and sample content of the Kilifi County Mental Health survey dataset using the Croissant metadata schema and `mlcroissant`. We:
- Loaded the dataset and inspected record sets and field IDs referenced by their `@id`s.
- Examined and loaded a record set into a DataFrame.
- Performed EDA including filtering, normalization, and grouping with explicit reference to `@id` fields.
- Visualized distributions and group differences.

This approach ensures reproducibility and transparency when working with FAIR data resources described by Croissant metadata.